# Crystalite — Property-Conditioned Generation (CFG) on Colab

Adapter fine-tunes the released MP20 base checkpoint to condition generation on
`band_gap` + `space_group`, then samples with classifier-free guidance.

**Runtime → Change runtime type → GPU (T4 is fine).**

Setup uses the repo's own `uv` + lockfile, so it installs the correct Python 3.12
and CUDA PyTorch regardless of Colab's defaults.

## 1. Clone the repo (branch with the CFG changes)

In [4]:
# If the repo is PRIVATE, replace the URL with:
#   https://<YOUR_GITHUB_TOKEN>@github.com/Blizzard57/crystalite-cfg.git
!git clone --branch cfg-optimized https://github.com/Blizzard57/crystalite-cfg.git
%cd crystalite-cfg

Cloning into 'crystalite-cfg'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 192 (delta 79), reused 143 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 326.67 KiB | 9.90 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/crystalite-cfg


## 2. Install dependencies with `uv` (a few minutes; downloads CUDA torch)

In [5]:
# !cd /Users/blizzard/Documents/Research/crystalite-cfg #&& git pull origin cfg-optimized
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ['PATH']
!uv python install 3.12
!uv sync

downloading uv 0.11.17 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Python 3.12 is already installed
Using CPython 3.12.13
Creating virtual environment at: .venv
Resolved 186 packages in 1ms
Prepared 179 packages in 1m 10s                                          
Installed 179 packages in 280ms                             
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiosignal==1.4.0
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.12.1
 + ase==3.27.0
 + asttokens==3.0.1
 + attrs==25.4.0
 + blosc2==4.1.2
 + cairocffi==1.7.1
 + cairosvg==2.8.2
 + certifi==2026.1.4
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + chgnet==0.4.2
 + click==8.3.1
 + cloudpickle==3.1.2
 + comm==0.2.3
 + contourpy==1.3.2
 + cssselect2==0.8.0
 + cycler==0.12.1
 + cython==3.2.4
 + datasets==4.4.2
 + debugpy==1.8.19
 + decorator==5.2.1
 + defusedxml==0.7.1
 + dill==0.4.0
 + dnspython==2.8.0
 + e3nn==0.6.0
 + executing==2.2.1
 + filelock==3.20.3


In [6]:
# Sanity check: GPU visible + our conditioning tests pass.
!uv run python -c "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"
!uv run --group dev python -m pytest tests/test_property_conditioning.py -q

cuda: True NVIDIA A100-SXM4-40GB
....                                                                     [100%]
4 passed in 2.54s


## 3. Download the MP20 dataset and the base checkpoint

In [7]:
# MP20 -> data/mp20/raw/{train,val,test}.csv
!uv run python src/data/download_datasets.py --datasets mp20 --out data

# Released DNG base checkpoint (MP20, subatomic_tokenizer_pca_16) -> ./best.pt
!uv run python -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id='joshrosie/crystalite-datasets', filename='best.pt', repo_type='dataset', local_dir='.')"

Repo: jbungle/crystalite-datasets
Output dir: /content/crystalite-cfg/data
Selected: mp20

Fetching 3 files:   0% 0/3 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 3 files: 100% 3/3 [00:01<00:00,  2.33it/s]
Download complete: 100% 135M/135M [00:01<00:00, 63.2MB/s]                   done -> /content/crystalite-cfg/data

Finished.
Download complete: 100% 135M/135M [00:01<00:00, 103MB/s] 
best.pt: 100% 541M/541M [00:04<00:00, 113MB/s]  


In [8]:
# Confirm the base checkpoint's architecture flags. The adapter fine-tune below MUST
# match these, or the strict=False load raises a clear mismatch error.
!uv run python -c "import torch; a=torch.load('best.pt', map_location='cpu', weights_only=False)['model_args']; keys=['type_encoding','d_model','n_heads','n_layers','lattice_repr','lattice_embed_mode','use_edge_bias','edge_bias_n_freqs','edge_bias_hidden_dim','edge_bias_n_rbf','coord_embed_mode','coord_n_freqs','sigma_data_type','sigma_data_coord','sigma_data_lattice']; print({k:a.get(k) for k in keys})"

{'type_encoding': 'subatomic_tokenizer_pca_16', 'd_model': 512, 'n_heads': 16, 'n_layers': 14, 'lattice_repr': 'ltri', 'lattice_embed_mode': 'mlp', 'use_edge_bias': True, 'edge_bias_n_freqs': 12, 'edge_bias_hidden_dim': 256, 'edge_bias_n_rbf': 32, 'coord_embed_mode': 'fourier', 'coord_n_freqs': 32, 'sigma_data_type': 0.3, 'sigma_data_coord': 0.3, 'sigma_data_lattice': 0.3}


## 4. Adapter fine-tune with conditioning (short demo: 200 steps)

The conditioner is zero-initialized, so training starts exactly at the base model and
learns the property dependence. `--sample_frequency 0` skips train-time eval for speed.
Bump `--max_steps` for real runs.

In [9]:
!uv run python src/train_crystalite.py \
  --data_root data/mp20 --dataset_name mp20 \
  --output_dir outputs/cond_mp20 \
  --nmax 20 --batch_size 64 --bf16 \
  --type_encoding subatomic_tokenizer_pca_16 \
  --d_model 512 --n_heads 16 --n_layers 14 \
  --use_edge_bias --edge_bias_n_freqs 12 --edge_bias_hidden_dim 256 --edge_bias_n_rbf 32 \
  --lattice_embed_mode mlp --lattice_repr ltri \
  --loss_weights 16 150 5 --coord_loss_mode frac_mse \
  --sigma_data_type 0.3 --sigma_data_coord 0.3 --sigma_data_lattice 0.3 \
  --cond_properties band_gap space_group \
  --cond_p_uncond 0.1 \
  --adapter_pretrained best.pt \
  --max_steps 200 --sample_frequency 0 --no_wandb

!ls -la outputs/cond_mp20/checkpoints

/content/crystalite-cfg/.venv/lib/python3.12/site-packages/nvidia_smi.py:810: SyntaxWarning: invalid escape sequence '\A'
  mem = 'N\A'
/content/crystalite-cfg/.venv/lib/python3.12/site-packages/nvidia_smi.py:831: SyntaxWarning: invalid escape sequence '\A'
  maxMemoryUsage = 'N\A'
[seed] seed=123 deterministic=False
100% 27138/27138 [02:07<00:00, 212.35it/s]
100% 9046/9046 [00:43<00:00, 207.30it/s]
Dataset split sizes (mp20, nmax=20): train=27138 val=9046
[eval] Metric references will reuse the training dataset (mp20 @ data/mp20).
Traceback (most recent call last):
  File "/content/crystalite-cfg/src/train_crystalite.py", line 857, in <module>
    main()
  File "/content/crystalite-cfg/src/train_crystalite.py", line 299, in main
    _setup_property_conditioning(model, args, ds, cond_prop_columns, device)
  File "/content/crystalite-cfg/src/train_crystalite.py", line 141, in _setup_property_conditioning
    model.conditioner.fit_stats(stats)
  File "/content/crystalite-cfg/src/models/p

## 5. Generate crystals conditioned on target properties (with CFG)

`--target NAME=VALUE` (repeatable) sets the conditioning; `--guidance_scale` controls
CFG strength (1.0 = plain conditional, >1 = stronger steering, 0 = unconditional).

In [10]:
!uv run python src/sample_crystalite_ckpt.py \
  --checkpoint outputs/cond_mp20/checkpoints/final.pt \
  --dataset_name mp20 --data_root data/mp20 --nmax 20 \
  --num_samples 64 --sample_chunk_size 64 --sample_num_steps 100 \
  --sample_mode regular --atom_count_strategy empirical \
  --bf16 --device cuda \
  --target band_gap=2.0 --target space_group=225 \
  --guidance_scale 2.0 \
  --output_dir outputs/cond_mp20/samples --save_cifs --cif_limit 64

!ls -la outputs/cond_mp20/samples

Traceback (most recent call last):
  File "/content/crystalite-cfg/src/sample_crystalite_ckpt.py", line 666, in <module>
    main()
  File "/content/crystalite-cfg/src/sample_crystalite_ckpt.py", line 380, in main
    ckpt_path = _resolve_checkpoint_path(
                ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/crystalite-cfg/src/sample_crystalite_ckpt.py", line 57, in _resolve_checkpoint_path
    raise FileNotFoundError(f"Checkpoint not found: {path}")
FileNotFoundError: Checkpoint not found: outputs/cond_mp20/checkpoints/final.pt
ls: cannot access 'outputs/cond_mp20/samples': No such file or directory


## Notes

- **If step 4 errors with `Adapter load mismatch`:** a train flag doesn't match `best.pt`.
  Re-run the step-3 cell and align the flags to the printed `model_args`.
- **Richer properties** (`dft_bulk_modulus`, `dft_mag_density`, `hhi_score`,
  `chemical_system`) live in **alex_mp20**, not MP20. Swap to `--datasets alex_mp20` /
  `--dataset_name alex_mp20 --data_root data/alex_mp20` and add them to
  `--cond_properties`. Heads-up: alex_mp20 is ~700 MB and first-time preprocessing is slow.
- **Scaling up:** increase `--max_steps`, enable `--ema_decay 0.9999` (then sample with
  `--sample_mode ema`), and raise `--sample_num_steps` (e.g. 150) for quality.
- **Compare guidance:** re-run step 5 with `--guidance_scale 0` vs `2.0` / `4.0` at the
  same `--sample_seed` to see the target take effect.